# P2 Build a codec

In [ ]:
import heapq
from collections import Counter

class Node:
	def __init__(self, symbol=None, frequency=None, order=0):
		self.symbol = symbol
		self.frequency = frequency
		self.order = order
		self.left = None
		self.right = None
	def __lt__(self, other):
		if self.frequency != other.frequency:
			return self.frequency < other.frequency
		return self.order < other.order

def build_huffman_tree(chars, freq):
	order = 0
	pri_queue = []

	for char, f in zip(chars, freq):
		heapq.heappush(pri_queue, Node(char, f, order))
		order += 1

	while len(pri_queue) > 1:
		left = heapq.heappop(pri_queue)
		right = heapq.heappop(pri_queue)
		merged = Node(frequency=left.frequency + right.frequency, order = order)
		merged.left = left
		merged.right = right
		heapq.heappush(pri_queue, merged)

	return pri_queue[0]

def get_code_lengths_from_tree(root):
	lengths = {}

	def walk(node, depth=0):
		if node.symbol is not None:
			lengths[node.symbol] = depth
		else:
			walk(node.left, depth + 1)
			walk(node.right, depth + 1)
	walk(root)

	if len(lengths) == 1:
		lengths[next(iter(lengths))] = 1

	return lengths

def huffman_lengths(counts):
	if not counts:
		return {}

	root = build_huffman_tree(counts)
	return get_code_lengths_from_tree(root)

def canonical_codes(present, lengths):
	present = sorted(present, key=lambda s: (lengths[s], s))
	codes = {}
	code = 0
	prev_len = 0

	for s in present:
		L = lengths[s]
		code <<= (L - prev_len)
		codes[s] = f"{code:0{L}b}"
		code += 1
		prev_len = L

	return codes

In [ ]:
MAGIC = b"HC\x01"

def compress_bytes(data):
	if not isinstance(data, (bytes, bytearray)):
		raise TypeError("compress expects bytes")

	data = bytes(data)

	if not data:
		return MAGIC + (0).to_bytes(8, "big") + (0).to_bytes(2, "big")

	freq = Counter(data)
	lengths = huffman_lengths(freq)
	codes = canonical_codes(lengths)
	present = sorted(freq)

	if len(freq) == 1:
		lengths = {data[0]: 1}
	else:
		root = build_huffman_tree(present, [freq[s] for s in present])
		lengths = get_code_lengths_from_tree(root)

	codes = canonical_codes(present, lengths)

	header = bytearray(MAGIC)             
	header += len(data).to_bytes(8, "big")    
	header += len(present).to_bytes(2, "big")
	for s in present:
		header += bytes([s, lengths[s]])

	return bytes(header) + codes
